In [8]:
%pip install polars pyarrow

  Using cached polars-1.44.2-py3-none-any.whl.metadata (11 kB)
  Using cached polars_runtime_32-1.44.2-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached polars-1.44.2-py3-none-any.whl (865 kB)
Using cached polars_runtime_32-1.44.2-cp310-abi3-win_amd64.whl (51.3 MB)

   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14000 entries, 0 to 13999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   signal_id      14000 non-null  object
 1   signal_sanasi  14000 non-null  object
 2   eskalatsiya    14000 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 328.3+ KB


In [9]:
import polars as pl

train_tx = pl.read_parquet("data/train_transactions.parquet")
test_tx = pl.read_parquet("data/test_transactions.parquet")

print(train_tx.shape)
print(test_tx.shape)

(6987663, 5)
(3027575, 5)


Schema({'signal_id': String, 'tranzaksiya_vaqti': Datetime(time_unit='us', time_zone=None), 'kirim_chiqim': String, 'tranzaksiya_turi': String, 'miqdor_indeksi': Float64})
Schema({'signal_id': String, 'tranzaksiya_vaqti': Datetime(time_unit='us', time_zone=None), 'kirim_chiqim': String, 'tranzaksiya_turi': String, 'miqdor_indeksi': Float64})


shape: (9, 3)
┌────────────┬───────────┬───────────────────┐
│ statistic  ┆ signal_id ┆ transaction_count │
│ ---        ┆ ---       ┆ ---               │
│ str        ┆ str       ┆ f64               │
╞════════════╪═══════════╪═══════════════════╡
│ count      ┆ 14000     ┆ 14000.0           │
│ null_count ┆ 0         ┆ 0.0               │
│ mean       ┆ null      ┆ 499.118786        │
│ std        ┆ null      ┆ 299.465434        │
│ min        ┆ SG_000002 ┆ 1.0               │
│ 25%        ┆ null      ┆ 277.0             │
│ 50%        ┆ null      ┆ 461.0             │
│ 75%        ┆ null      ┆ 678.0             │
│ max        ┆ SG_019997 ┆ 2279.0            │
└────────────┴───────────┴───────────────────┘


signal_id,transaction_count
str,u32
"""SG_013858""",2279
"""SG_005198""",2129
"""SG_000410""",2088
"""SG_007152""",2039
"""SG_019605""",2019
…,…
"""SG_017237""",1743
"""SG_012478""",1732
"""SG_016162""",1714


eskalatsiya
0    11595
1     2405
Name: count, dtype: int64

In [34]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [42]:


import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier




train_signals = pd.read_csv(
    "data/train_signals.csv"
)

test_signals = pd.read_csv(
    "data/test_signals.csv"
)

train_transactions = pd.read_parquet(
    "data/train_transactions.parquet"
)

test_transactions = pd.read_parquet(
    "data/test_transactions.parquet"
)

sample_submission = pd.read_csv(
    "data/sample_submission.csv"
)


print("TRAIN SIGNALS:", train_signals.shape)
print("TEST SIGNALS:", test_signals.shape)

print("TRAIN TRANSACTIONS:", train_transactions.shape)
print("TEST TRANSACTIONS:", test_transactions.shape)

print("SAMPLE SUBMISSION:", sample_submission.shape)



train_signals["signal_sanasi"] = pd.to_datetime(
    train_signals["signal_sanasi"]
)

test_signals["signal_sanasi"] = pd.to_datetime(
    test_signals["signal_sanasi"]
)

train_transactions["tranzaksiya_vaqti"] = pd.to_datetime(
    train_transactions["tranzaksiya_vaqti"]
)

test_transactions["tranzaksiya_vaqti"] = pd.to_datetime(
    test_transactions["tranzaksiya_vaqti"]
)




def create_features(signals, transactions):



    tx = transactions.merge(
        signals[["signal_id", "signal_sanasi"]],
        on="signal_id",
        how="inner"
    )



    tx = tx[
        tx["tranzaksiya_vaqti"] < tx["signal_sanasi"]
    ].copy()



    tx["days_before"] = (
        tx["signal_sanasi"]
        - tx["tranzaksiya_vaqti"]
    ).dt.total_seconds() / 86400



    tx["is_in"] = (
        tx["kirim_chiqim"] == "kirim"
    ).astype("int8")

    tx["is_out"] = (
        tx["kirim_chiqim"] == "chiqim"
    ).astype("int8")

    tx["is_card"] = (
        tx["tranzaksiya_turi"] == "karta"
    ).astype("int8")

    tx["is_bank"] = (
        tx["tranzaksiya_turi"] == "bank_otkazmasi"
    ).astype("int8")

    tx["is_cash"] = (
        tx["tranzaksiya_turi"] == "naqd"
    ).astype("int8")

    tx["is_international"] = (
        tx["tranzaksiya_turi"] == "xalqaro"
    ).astype("int8")



    features = (
        tx.groupby("signal_id")
        .agg(
            transaction_count=(
                "signal_id",
                "size"
            ),

            amount_sum=(
                "miqdor_indeksi",
                "sum"
            ),

            amount_mean=(
                "miqdor_indeksi",
                "mean"
            ),

            amount_std=(
                "miqdor_indeksi",
                "std"
            ),

            amount_min=(
                "miqdor_indeksi",
                "min"
            ),

            amount_max=(
                "miqdor_indeksi",
                "max"
            ),

            first_transaction=(
                "tranzaksiya_vaqti",
                "min"
            ),

            last_transaction=(
                "tranzaksiya_vaqti",
                "max"
            ),

            active_days=(
                "tranzaksiya_vaqti",
                lambda x: x.dt.date.nunique()
            ),
        )
    )



    quantiles = (
        tx.groupby("signal_id")["miqdor_indeksi"]
        .quantile([0.25, 0.50, 0.75, 0.90])
        .unstack()
        .rename(
            columns={
                0.25: "amount_q25",
                0.50: "amount_median",
                0.75: "amount_q75",
                0.90: "amount_q90"
            }
        )
    )

    features = features.join(
        quantiles,
        how="left"
    )



    direction_counts = (
        tx.groupby("signal_id")
        .agg(
            incoming_count=("is_in", "sum"),
            outgoing_count=("is_out", "sum")
        )
    )

    features = features.join(
        direction_counts,
        how="left"
    )



    tx["incoming_amount"] = np.where(
        tx["is_in"] == 1,
        tx["miqdor_indeksi"],
        0
    )

    tx["outgoing_amount"] = np.where(
        tx["is_out"] == 1,
        tx["miqdor_indeksi"],
        0
    )

    direction_amounts = (
        tx.groupby("signal_id")
        .agg(
            incoming_amount=(
                "incoming_amount",
                "sum"
            ),

            outgoing_amount=(
                "outgoing_amount",
                "sum"
            )
        )
    )

    features = features.join(
        direction_amounts,
        how="left"
    )


    type_counts = (
        tx.groupby("signal_id")
        .agg(
            card_count=("is_card", "sum"),
            bank_transfer_count=("is_bank", "sum"),
            cash_count=("is_cash", "sum"),
            international_count=(
                "is_international",
                "sum"
            )
        )
    )

    features = features.join(
        type_counts,
        how="left"
    )



    tx["international_amount"] = np.where(
        tx["is_international"] == 1,
        tx["miqdor_indeksi"],
        0
    )

    tx["cash_amount"] = np.where(
        tx["is_cash"] == 1,
        tx["miqdor_indeksi"],
        0
    )

    type_amounts = (
        tx.groupby("signal_id")
        .agg(
            international_amount=(
                "international_amount",
                "sum"
            ),

            cash_amount=(
                "cash_amount",
                "sum"
            )
        )
    )

    features = features.join(
        type_amounts,
        how="left"
    )



    windows = [1, 3, 7, 14, 30, 60, 90]

    for days in windows:

        recent = tx[
            tx["days_before"] <= days
        ]

        if len(recent) == 0:
            continue

        recent_features = (
            recent.groupby("signal_id")
            .agg(
                **{
                    f"transactions_last_{days}d":
                        ("signal_id", "size"),

                    f"amount_last_{days}d":
                        ("miqdor_indeksi", "sum"),

                    f"outgoing_last_{days}d":
                        ("is_out", "sum"),

                    f"incoming_last_{days}d":
                        ("is_in", "sum"),

                    f"international_last_{days}d":
                        ("is_international", "sum")
                }
            )
        )

        features = features.join(
            recent_features,
            how="left"
        )

 

    features["days_since_last_transaction"] = (
        features.index.to_series()
        .map(
            (
                tx.groupby("signal_id")
                ["days_before"]
                .min()
            )
        )
    )



    features["history_span_days"] = (
        features["last_transaction"]
        - features["first_transaction"]
    ).dt.total_seconds() / 86400



    signal_dates = (
        signals[
            [
                "signal_id",
                "signal_sanasi"
            ]
        ]
        .set_index("signal_id")
    )

    features = features.join(
        signal_dates,
        how="right"
    )

    features["signal_year"] = (
        features["signal_sanasi"].dt.year
    )

    features["signal_month"] = (
        features["signal_sanasi"].dt.month
    )

    features["signal_day"] = (
        features["signal_sanasi"].dt.day
    )

    features["signal_dayofweek"] = (
        features["signal_sanasi"].dt.dayofweek
    )

    features["signal_dayofyear"] = (
        features["signal_sanasi"].dt.dayofyear
    )



    denominator = (
        features["transaction_count"]
        .replace(0, np.nan)
    )

    features["incoming_ratio"] = (
        features["incoming_count"]
        / denominator
    )

    features["outgoing_ratio"] = (
        features["outgoing_count"]
        / denominator
    )

    features["international_ratio"] = (
        features["international_count"]
        / denominator
    )

    features["cash_ratio"] = (
        features["cash_count"]
        / denominator
    )

    features["card_ratio"] = (
        features["card_count"]
        / denominator
    )

    features["bank_transfer_ratio"] = (
        features["bank_transfer_count"]
        / denominator
    )


    total_amount = (
        features["amount_sum"]
        .replace(0, np.nan)
    )

    features["incoming_amount_ratio"] = (
        features["incoming_amount"]
        / total_amount
    )

    features["outgoing_amount_ratio"] = (
        features["outgoing_amount"]
        / total_amount
    )

    features["international_amount_ratio"] = (
        features["international_amount"]
        / total_amount
    )


    features["transactions_per_active_day"] = (
        features["transaction_count"]
        / features["active_days"].replace(0, np.nan)
    )


    for days in [3, 7, 14, 30]:

        col = f"transactions_last_{days}d"

        if col in features.columns:

            features[
                f"recent_{days}d_ratio"
            ] = (
                features[col]
                / denominator
            )



    if (
        "transactions_last_7d" in features.columns
        and
        "transactions_last_30d" in features.columns
    ):

        features["activity_7d_vs_30d"] = (
            features["transactions_last_7d"]
            /
            features["transactions_last_30d"]
            .replace(0, np.nan)
        )

    if (
        "transactions_last_3d" in features.columns
        and
        "transactions_last_30d" in features.columns
    ):

        features["activity_3d_vs_30d"] = (
            features["transactions_last_3d"]
            /
            features["transactions_last_30d"]
            .replace(0, np.nan)
        )



    features = features.drop(
        columns=[
            "signal_sanasi",
            "first_transaction",
            "last_transaction"
        ],
        errors="ignore"
    )



    features = features.reset_index()



    features = features.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return features



print("\nCreating training features...")

train_features = create_features(
    train_signals,
    train_transactions
)

print(
    "Training feature shape:",
    train_features.shape
)




print("\nCreating test features...")

test_features = create_features(
    test_signals,
    test_transactions
)

print(
    "Testing feature shape:",
    test_features.shape
)




train_features = train_features.merge(
    train_signals[
        [
            "signal_id",
            "eskalatsiya"
        ]
    ],
    on="signal_id",
    how="left"
)



print("\nTARGET DISTRIBUTION")

print(
    train_features["eskalatsiya"].value_counts()
)

print(
    train_features["eskalatsiya"]
    .value_counts(normalize=True)
)



drop_columns = [
    "signal_id",
    "eskalatsiya"
]

X = train_features.drop(
    columns=drop_columns
)

y = train_features["eskalatsiya"]

X_test = test_features.drop(
    columns=["signal_id"]
)



X_test = X_test.reindex(
    columns=X.columns
)


print("\nFEATURE COUNT:", X.shape[1])




cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


oof_predictions = np.zeros(
    len(X)
)

test_predictions = np.zeros(
    len(X_test)
)



for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    print(
        f"\n================ FOLD {fold} ================"
    )

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(

       
        iterations=2500,

        learning_rate=0.03,

       
        depth=7,

       
        l2_leaf_reg=5,

       
        eval_metric="AUC",

        
        loss_function="Logloss",

        
        early_stopping_rounds=150,

       
        random_seed=42 + fold,

        
        verbose=False,

        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        verbose=False
    )


    valid_probability = model.predict_proba(
        X_valid
    )[:, 1]

 

    oof_predictions[valid_idx] = (
        valid_probability
    )



    fold_auc = roc_auc_score(
        y_valid,
        valid_probability
    )

    print(
        f"Fold {fold} ROC-AUC: {fold_auc:.6f}"
    )



    test_predictions += (
        model.predict_proba(X_test)[:, 1]
        / cv.n_splits
    )




overall_auc = roc_auc_score(
    y,
    oof_predictions
)

print("\n======================================")
print("OVERALL CV ROC-AUC:", overall_auc)
print("======================================")




predictions = pd.DataFrame({

    "signal_id":
        test_signals["signal_id"],

    "ehtimollik":
        test_predictions

})




predictions["ehtimollik"] = predictions[
    "ehtimollik"
].clip(
    0,
    1
)




print(
    "\nDuplicate IDs:",
    predictions["signal_id"].duplicated().sum()
)



print(
    "Expected test signals:",
    len(test_signals)
)

print(
    "Predicted signals:",
    len(predictions)
)




assert set(
    predictions["signal_id"]
) == set(
    test_signals["signal_id"]
)

print(
    "All test signal IDs are present."
)




print(
    "Minimum probability:",
    predictions["ehtimollik"].min()
)

print(
    "Maximum probability:",
    predictions["ehtimollik"].max()
)

assert (
    predictions["ehtimollik"].between(0, 1).all()
)

print(
    "All probabilities are between 0 and 1."
)




print(
    "\nSample submission columns:",
    sample_submission.columns.tolist()
)

print(
    "Our columns:",
    predictions.columns.tolist()
)

assert sample_submission.columns.tolist() == [
    "signal_id",
    "ehtimollik"
]

assert predictions.columns.tolist() == [
    "signal_id",
    "ehtimollik"
]

print(
    "Submission format is correct."
)




top_alerts = predictions.sort_values(
    "ehtimollik",
    ascending=False
)

print(
    "\nTOP 30 HIGHEST-PROBABILITY ALERTS"
)

display(
    top_alerts.head(30)
)




print(
    "\nBOTTOM 30 LOWEST-PROBABILITY ALERTS"
)

display(
    top_alerts.tail(30)
)




print("\nPROBABILITY STATISTICS")

print(
    predictions["ehtimollik"].describe()
)



TEAM_ID = "Myst"

submission = predictions[
    ["signal_id", "ehtimollik"]
].copy()

submission["ehtimollik"] = submission["ehtimollik"].clip(0, 1)

assert submission.columns.tolist() == [
    "signal_id",
    "ehtimollik"
]

assert len(submission) == len(test_signals)
assert submission["signal_id"].duplicated().sum() == 0
assert submission["signal_id"].notna().all()
assert submission["ehtimollik"].notna().all()
assert set(submission["signal_id"]) == set(test_signals["signal_id"])
assert submission["ehtimollik"].between(0, 1).all()

filename = f"team_{TEAM_ID}.csv"

submission.to_csv(
    filename,
    index=False,
    float_format="%.6f"
)

print(f"FINAL FILE CREATED: {filename}")
print("\nShape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Duplicate IDs:", submission["signal_id"].duplicated().sum())
print("Missing predictions:", submission["ehtimollik"].isna().sum())
print("Minimum probability:", submission["ehtimollik"].min())
print("Maximum probability:", submission["ehtimollik"].max())

display(submission.head(20))

TRAIN SIGNALS: (14000, 3)
TEST SIGNALS: (6000, 2)
TRAIN TRANSACTIONS: (6987663, 5)
TEST TRANSACTIONS: (3027575, 5)
SAMPLE SUBMISSION: (6000, 2)

Creating training features...
Training feature shape: (14000, 80)

Creating test features...
Testing feature shape: (6000, 80)

TARGET DISTRIBUTION
eskalatsiya
0    11595
1     2405
Name: count, dtype: int64
eskalatsiya
0    0.828214
1    0.171786
Name: proportion, dtype: float64

FEATURE COUNT: 79

================ FOLD 1 ================
Fold 1 ROC-AUC: 0.608404

================ FOLD 2 ================
Fold 2 ROC-AUC: 0.595810

================ FOLD 3 ================
Fold 3 ROC-AUC: 0.600025

================ FOLD 4 ================
Fold 4 ROC-AUC: 0.602059

================ FOLD 5 ================
Fold 5 ROC-AUC: 0.587862

OVERALL CV ROC-AUC: 0.5986670360279674

Duplicate IDs: 0
Expected test signals: 6000
Predicted signals: 6000
All test signal IDs are present.
Minimum probability: 0.05332211013690795
Maximum probability: 0.3756915158539

,signal_id,ehtimollik
5601,SG_018710,0.375692
3339,SG_011139,0.344742
4545,SG_015297,0.343887
1907,SG_006266,0.332984
5407,SG_018124,0.331139
2765,SG_009226,0.328515
82,SG_000290,0.326123
2552,SG_008534,0.324369
2123,SG_006982,0.320950
4272,SG_014311,0.319476



BOTTOM 30 LOWEST-PROBABILITY ALERTS


,signal_id,ehtimollik
1733,SG_005722,0.072659
4262,SG_014272,0.072378
126,SG_000431,0.072273
779,SG_002601,0.071333
3403,SG_011340,0.070376
4836,SG_016187,0.069875
1328,SG_004462,0.069622
4886,SG_016332,0.069565
4186,SG_014018,0.069322
3191,SG_010589,0.069225



PROBABILITY STATISTICS
count    6000.000000
mean        0.169291
std         0.043425
min         0.053322
25%         0.139572
50%         0.164526
75%         0.197053
max         0.375692
Name: ehtimollik, dtype: float64
FINAL FILE CREATED: team_Myst.csv

Shape: (6000, 2)
Columns: ['signal_id', 'ehtimollik']
Duplicate IDs: 0
Missing predictions: 0
Minimum probability: 0.05332211013690795
Maximum probability: 0.37569151585390276


,signal_id,ehtimollik
0,SG_000001,0.157951
1,SG_000007,0.198103
2,SG_000009,0.126584
3,SG_000010,0.121739
4,SG_000011,0.125370
5,SG_000018,0.193364
6,SG_000024,0.183852
7,SG_000026,0.155781
8,SG_000032,0.207150
9,SG_000033,0.206886


NameError: name 'train_features' is not defined

,signal_id,ehtimollik
5601,SG_018710,0.375692
3339,SG_011139,0.344742
4545,SG_015297,0.343887
1907,SG_006266,0.332984
5407,SG_018124,0.331139
2765,SG_009226,0.328515
82,SG_000290,0.326123
2552,SG_008534,0.324369
2123,SG_006982,0.320950
4272,SG_014311,0.319476


,signal_id,ehtimollik
5775,SG_019298,0.065025
5218,SG_017480,0.064843
36,SG_000107,0.064755
1649,SG_005459,0.064244
3703,SG_012378,0.062142
4335,SG_014569,0.061123
3609,SG_012079,0.056701
5193,SG_017402,0.056569
1038,SG_003489,0.055412
1325,SG_004458,0.053322


In [40]:
import pandas as pd

# Load existing file
df = pd.read_csv("team_Myst.csv")

# Sort by probability (highest first)
df = df.sort_values(by="ehtimollik", ascending=False)
print(df)
# Overwrite the file


print("team_Myst.csv successfully sorted!")

      signal_id  ehtimollik
0     SG_018710    0.375692
1     SG_011139    0.344742
2     SG_015297    0.343887
3     SG_006266    0.332984
4     SG_018124    0.331139
...         ...         ...
5995  SG_014569    0.061123
5996  SG_012079    0.056701
5997  SG_017402    0.056569
5998  SG_003489    0.055412
5999  SG_004458    0.053322

[6000 rows x 2 columns]
team_Myst.csv successfully sorted!
